# Notebook 13 — Capstone: A Complete Study of a Molecule

**MOLEKUL | all phases, end to end**

---

The previous twelve notebooks each focused on one piece of machinery. This capstone
puts them together the way you actually would in a research workflow: take a molecule,
find its real structure, prove that structure is stable, then characterise it — charges,
spectrum, correlation energy, excitations — and write up the result.

We will study **water** in the STO-3G basis (small enough to run in a few minutes,
rich enough to show every step). The pipeline is:

1. Define the system (from an XYZ string).
2. A first single point (RHF).
3. **Optimise** the geometry to the true minimum.
4. **Confirm** it is a minimum (harmonic frequencies, no imaginary modes).
5. Electronic **properties**: Mulliken charges and dipole.
6. **Correlation**: how much energy HF missed (MP2, CCSD).
7. A **DFT** cross-check (LDA).
8. **Excited states** (CIS and TD-DFT).
9. Assemble the **report**.

Everything below is computed live — nothing is hard-coded.

In [1]:
# --- Make the MOLEKUL package importable -------------------------------
# Best practice: install once from the repo root with
#     pip install -e ".[notebooks]"
# The fallback below locates the in-repo src/ automatically, so the
# notebook also runs from a fresh clone that has not been installed yet,
# regardless of which directory Jupyter was started from.
try:
    import molekul  # noqa: F401
except ModuleNotFoundError:
    import sys, pathlib
    for _p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
        if (_p / "src" / "molekul").is_dir():
            sys.path.insert(0, str(_p / "src"))
            break
    import molekul  # noqa: F401
# -----------------------------------------------------------------------

import numpy as np

from molekul.atoms import Atom
from molekul.molecule import Molecule
from molekul.io_xyz import read_xyz
from molekul.basis_sto3g import STO3G
from molekul.rhf import rhf_scf
from molekul.optimizer import optimize_geometry
from molekul.freqs import harmonic_analysis
from molekul.integrals import build_overlap
from molekul.population import mulliken_populations, dipole_moment
from molekul.mp2 import mp2_energy
from molekul.ccsd import ccsd_energy, ccsdt_energy
from molekul.dft import ks_scf
from molekul.cis import cis_excitations
from molekul.tddft import tddft_tda
from molekul.constants import BOHR_TO_ANGSTROM, HARTREE_TO_KCAL_MOL, HARTREE_TO_EV

basis = STO3G
print("Ready.")

Ready.


## Step 1 — Define the system

We start, as any study does, from a guessed geometry. To make the optimisation do
visible work we deliberately start from a *distorted* water molecule (stretched bonds,
wrong angle) given as an XYZ string.

In [2]:
import tempfile, os

xyz_text = """3
distorted water (deliberately off)
O   0.0000   0.0000   0.2000
H   0.9000   0.0000  -0.5000
H  -0.9000   0.0000  -0.5000
"""
with tempfile.NamedTemporaryFile("w", suffix=".xyz", delete=False) as f:
    f.write(xyz_text)
    _tmp = f.name
mol_start = read_xyz(_tmp)
os.unlink(_tmp)

def bond_A(m, i, j):
    return np.linalg.norm(m.atoms[i].coords - m.atoms[j].coords) * BOHR_TO_ANGSTROM

def angle_deg(m, i, j, k):
    v1 = m.atoms[i].coords - m.atoms[j].coords
    v2 = m.atoms[k].coords - m.atoms[j].coords
    return np.degrees(np.arccos(np.clip(v1 @ v2 / (np.linalg.norm(v1)*np.linalg.norm(v2)), -1, 1)))

print(mol_start)
print(f"start:  r(O-H) = {bond_A(mol_start,0,1):.4f} A,  angle = {angle_deg(mol_start,1,0,2):.2f} deg")

Molecule(name='distorted water (deliberately off)', charge=0, mult=1)
  Atom(O, Z=8, xyz=[0.000000, 0.000000, 0.200000] Å)
  Atom(H, Z=1, xyz=[0.900000, 0.000000, -0.500000] Å)
  Atom(H, Z=1, xyz=[-0.900000, 0.000000, -0.500000] Å)
  n_electrons=10, n_alpha=5, n_beta=5
start:  r(O-H) = 1.1402 A,  angle = 104.25 deg


## Step 2 — A first single point (RHF)

Before optimising, one RHF calculation at the starting geometry tells us the energy of
the structure we guessed.

In [3]:
rhf_start = rhf_scf(mol_start, basis)
print(f"RHF energy at the starting (distorted) geometry: {rhf_start.energy_total:.6f} Ha")

RHF energy at the starting (distorted) geometry: -74.925328 Ha


## Step 3 — Optimise the geometry

`optimize_geometry` walks downhill on the Born-Oppenheimer surface (BFGS, numerical
gradient) until the forces vanish. The result is the equilibrium structure.

In [4]:
opt = optimize_geometry(mol_start, basis, verbose=False)
mol = opt.final_molecule   # the optimised molecule — everything below uses this

print(f"converged : {opt.converged} in {opt.n_steps} steps")
print(f"E_start   : {opt.energy_initial:.6f} Ha")
print(f"E_final   : {opt.energy_final:.6f} Ha")
print(f"relaxation: {(opt.energy_final-opt.energy_initial)*HARTREE_TO_KCAL_MOL:.2f} kcal/mol")
print()
print(f"optimised : r(O-H) = {bond_A(mol,0,1):.4f} A  (exp. 0.958 A)")
print(f"            angle  = {angle_deg(mol,1,0,2):.2f} deg  (exp. 104.5 deg)")

converged : True in 7 steps
E_start   : -74.925328 Ha
E_final   : -74.965901 Ha
relaxation: -25.46 kcal/mol

optimised : r(O-H) = 0.9894 A  (exp. 0.958 A)
            angle  = 100.02 deg  (exp. 104.5 deg)


## Step 4 — Confirm it is a minimum

A vanishing gradient alone does not prove a minimum — it could be a saddle point.
The harmonic frequencies settle it: a true minimum has **no imaginary frequencies**.
We also get the zero-point energy and a simulated IR stick spectrum for free.

In [5]:
freq = harmonic_analysis(mol, basis, verbose=False)

print("Vibrational frequencies (cm^-1):")
for k, (w, A) in enumerate(zip(freq.frequencies, freq.intensities), 1):
    print(f"  mode {k}: {w:8.1f}   IR intensity {A:7.1f} km/mol")
print(f"\nimaginary modes : {freq.n_imaginary}   ->  {'MINIMUM confirmed' if freq.n_imaginary == 0 else 'NOT a minimum!'}")
print(f"zero-point energy: {freq.zero_point_energy*HARTREE_TO_KCAL_MOL:.2f} kcal/mol")

Vibrational frequencies (cm^-1):
  mode 1:   2169.9   IR intensity     7.2 km/mol
  mode 2:   4139.8   IR intensity    44.3 km/mol
  mode 3:   4390.9   IR intensity    30.0 km/mol

imaginary modes : 0   ->  MINIMUM confirmed
zero-point energy: 15.30 kcal/mol


## Step 5 — Electronic properties

At the optimised geometry we read off chemically meaningful numbers: the Mulliken
atomic charges and the molecular dipole moment. (Recall from notebook 06 that Mulliken
charges are basis-dependent and only qualitative.)

In [6]:
rhf = rhf_scf(mol, basis)            # RHF at the optimised geometry
S   = build_overlap(basis, mol)
pop = mulliken_populations(rhf.density_matrix, S, basis, mol)
dip = dipole_moment(rhf.density_matrix, basis, mol)

print("Mulliken charges:")
for i, at in enumerate(mol.atoms):
    print(f"  {at.symbol}{i}: {pop.mulliken_charges[i]:+.3f} e")
print(f"\ndipole moment |mu| = {dip.magnitude_debye:.3f} D  (exp. 1.85 D)")

Mulliken charges:
  O0: -0.331 e
  H1: +0.165 e
  H2: +0.165 e

dipole moment |mu| = 1.709 D  (exp. 1.85 D)


## Step 6 — How much did Hartree-Fock miss?

RHF is a mean-field method: it ignores the *correlated* motion of electrons. MP2 and
CCSD recover that missing correlation energy. CCSD(T) — the "gold standard" — adds the
leading triples on top.

In [7]:
mp2   = mp2_energy(mol, basis, rhf)
ccsd  = ccsd_energy(mol, basis, rhf)
ccsdt = ccsdt_energy(mol, basis, rhf)

print(f"{'method':10s}{'E_total (Ha)':>16}{'E_corr (Ha)':>16}")
print("-" * 42)
print(f"{'RHF':10s}{rhf.energy_total:>16.6f}{0.0:>16.6f}")
print(f"{'MP2':10s}{mp2.energy_total:>16.6f}{mp2.energy_mp2:>16.6f}")
print(f"{'CCSD':10s}{ccsd.energy_total:>16.6f}{ccsd.energy_ccsd:>16.6f}")
print(f"{'CCSD(T)':10s}{ccsdt.energy_total:>16.6f}{ccsdt.energy_ccsdt:>16.6f}")
print(f"\nCCSD recovers {abs(ccsd.energy_ccsd)*HARTREE_TO_KCAL_MOL:.1f} kcal/mol of correlation energy.")

method        E_total (Ha)     E_corr (Ha)
------------------------------------------
RHF             -74.965901        0.000000
MP2             -75.004855       -0.038954
CCSD            -75.020284       -0.054383
CCSD(T)         -75.020351       -0.000067

CCSD recovers 34.1 kcal/mol of correlation energy.


## Step 7 — A DFT cross-check

Density functional theory reaches correlation-aware accuracy at roughly HF cost. We run
LDA (validated against PySCF to ~$10^{-5}$ Ha) as an independent check on the energy.

In [8]:
lda = ks_scf(mol, basis, xc="lda", verbose=False)
print(f"RHF      : {rhf.energy_total:.6f} Ha")
print(f"LDA      : {lda.energy_total:.6f} Ha")
print(f"CCSD(T)  : {ccsdt.energy_total:.6f} Ha   (best estimate here)")

RHF      : -74.965901 Ha
LDA      : -74.739878 Ha
CCSD(T)  : -75.020351 Ha   (best estimate here)


## Step 8 — Excited states

Finally, the optical fingerprint. CIS is the cheapest excited-state method; TD-DFT/TDA
is the affordable workhorse. (For quantitative excitation energies you would use
EOM-CCSD — notebook 10 — but it is the most expensive.)

In [9]:
cis = cis_excitations(mol, basis, rhf, n_states=3)
td  = tddft_tda(mol, basis, "lda", n_states=3)

print(f"{'state':>6}{'CIS (eV)':>12}{'TD-DFT/LDA (eV)':>18}")
print("-" * 36)
for i in range(3):
    print(f"S{i+1:>5}{cis.excitation_energies_ev[i]:>12.3f}{td.excitation_eV[i]:>18.3f}")


  CIS: n_occ=5, n_virt=2, dim=10
  Building MO ERIs …


  Building CIS matrix …
  Diagonalising …

  State       ΔE (Ha)     ΔE (eV)           f
  ----------------------------------------------
      1      0.458856     12.4861    0.003042
      2      0.515059     14.0155    0.000000
      3      0.604795     16.4573    0.081386


 state    CIS (eV)   TD-DFT/LDA (eV)
------------------------------------
S    1      12.486            10.823
S    2      14.015            12.711
S    3      16.457            14.143


## Step 9 — The report

Putting the whole study on one line per quantity — the kind of table that would open the
results section of a paper (with the caveat that STO-3G is a *minimal* basis, so absolute
numbers are illustrative, not publication-grade).

In [10]:
print("=" * 52)
print(" WATER - RHF/STO-3G study (optimised geometry)")
print("=" * 52)
print(f" r(O-H)              : {bond_A(mol,0,1):.4f} A")
print(f" angle H-O-H         : {angle_deg(mol,1,0,2):.2f} deg")
print(f" minimum confirmed   : {freq.n_imaginary == 0} (0 imaginary modes)")
print(f" zero-point energy   : {freq.zero_point_energy*HARTREE_TO_KCAL_MOL:.2f} kcal/mol")
print(f" dipole moment       : {dip.magnitude_debye:.3f} D")
print(f" Mulliken q(O)       : {pop.mulliken_charges[0]:+.3f} e")
print(f" E[RHF]              : {rhf.energy_total:.6f} Ha")
print(f" E[CCSD(T)]          : {ccsdt.energy_total:.6f} Ha")
print(f" correlation (CCSD(T)): {ccsdt.energy_ccsdt*HARTREE_TO_KCAL_MOL:.1f} kcal/mol")
print(f" first excitation S1 : {cis.excitation_energies_ev[0]:.2f} eV (CIS)")
print("=" * 52)

 WATER - RHF/STO-3G study (optimised geometry)
 r(O-H)              : 0.9894 A
 angle H-O-H         : 100.02 deg
 minimum confirmed   : True (0 imaginary modes)
 zero-point energy   : 15.30 kcal/mol
 dipole moment       : 1.709 D
 Mulliken q(O)       : -0.331 e
 E[RHF]              : -74.965901 Ha
 E[CCSD(T)]          : -75.020351 Ha
 correlation (CCSD(T)): -0.0 kcal/mol
 first excitation S1 : 12.49 eV (CIS)


---

## When *not* to use MOLEKUL — and where to go next

MOLEKUL is a **teaching** code: every line is meant to be read and understood. That
design choice has costs, and an honest study names them.

- **Size.** Two-electron integrals are stored densely ($N^4$), so the practical ceiling
  is ~50 basis functions — a handful of heavy atoms. No integral screening, no density
  fitting.
- **Elements.** Calculations are limited by basis coverage (STO-3G: H–Ne; 6-31G\*/cc-pVDZ:
  H, He, C, N, O, F).
- **Accuracy.** STO-3G is a minimal basis; for real numbers you need at least cc-pVDZ and
  usually extrapolation to the basis-set limit. A few methods here (PBE, PBE TD-DFT) are
  flagged experimental.
- **Speed.** Pure Python with finite-difference gradients and Hessians — fine for
  learning, far too slow for production.

**For real research** reach for a production package — they implement the same physics
you just read, at scale:

| Need | Tools |
|------|-------|
| Molecular HF / post-HF / DFT | PySCF, Psi4, ORCA, Gaussian, Q-Chem |
| Periodic solids | VASP, Quantum ESPRESSO, CP2K, GPAW |
| Very large / linear-scaling | CP2K, ONETEP, BigDFT |

The value of having read these notebooks is that none of those will be a black box to
you anymore: you have seen the overlap matrix, the DIIS error vector, the amplitude
equations, and the Bloch sum with your own eyes.

**Further reading:** Szabo & Ostlund, *Modern Quantum Chemistry*; Helgaker, Jørgensen &
Olsen, *Molecular Electronic-Structure Theory*; Martin, *Electronic Structure*.

---

*This is the final notebook in the MOLEKUL series. You have built a quantum chemistry
program from an atom to a phonon — congratulations.*